In [0]:
%fs ls abfss://labdata@dbdeltalakestorage.dfs.core.windows.net/invoices/

In [0]:
SELECT * FROM
PARQUET.`abfss://labdata@dbdeltalakestorage.dfs.core.windows.net/invoices/invoices_101_200.parquet`
LIMIT 5;

#### Copy On Write

In [0]:
CREATE or REPLACE Table deltacatalog.deltadb.invoices_cow 
TBLPROPERTIES('delta.enableDeletionVectors' = false)
AS 
Select * FROM
PARQUET.`abfss://labdata@dbdeltalakestorage.dfs.core.windows.net/invoices/invoices_101_200.parquet`

In [0]:
DESCRIBE EXTENDED deltacatalog.deltadb.invoices_cow;

Correct the age of user 105 from 57 to 55

In [0]:
UPDATE deltacatalog.deltadb.invoices_cow
SET age=55 
 where customer_id=105;

In [0]:
DESCRIBE HISTORY deltacatalog.deltadb.invoices_cow;

Understanding Operation Metrics :
1. numRemovedFiles: "1" 
2. numCopiedRows: "99"
3. numAddedFiles: "1"
4. numUpdatedRows: "1"
5. In Storage I can see two files now :
   - part-00000-89d9954b-1f56-45ed-835b-19049931a294.c000.zstd.parquet 02/05/2026, 04:50:08 4.61 KiB
   - part-00000-b62a1e0b-fe05-4fb5-809a-625d19c64afd.c000.zstd.parquet 02/05/2026, 04:55:50 4.59 KiB

In [0]:
DELETE FROM deltacatalog.deltadb.invoices_cow
where customer_id=102;

In [0]:
DESCRIBE HISTORY deltacatalog.deltadb.invoices_cow;

Understanding Operation Metrics :
1. numRemovedFiles: "1" 
2. numCopiedRows: "99"
3. numAddedFiles: "1"
4. numDeletedRows: "1"
5. In Storage I can see two files now :
   - part-00000-89d9954b-1f56-45ed-835b-19049931a294.c000.zstd.parquet 02/05/2026, 04:50:08 4.61 KiB
   - part-00000-b62a1e0b-fe05-4fb5-809a-625d19c64afd.c000.zstd.parquet 02/05/2026, 04:55:50 4.59 KiB
   - part-00000-6203cc35-bcaa-41c2-86e0-2a2b9f2a97b3.c000.zstd.parquet 02/05/2026, 05:03:19 4.59 KiB

#### MoR (Merge-On-Read)

In [0]:
CREATE or REPLACE Table deltacatalog.deltadb.invoices_mor 
TBLPROPERTIES('delta.enableDeletionVectors' = true)
AS 
Select * FROM
PARQUET.`abfss://labdata@dbdeltalakestorage.dfs.core.windows.net/invoices/invoices_101_200.parquet`

In [0]:
DESCRIBE EXTENDED deltacatalog.deltadb.invoices_mor;

In [0]:
DELETE FROM deltacatalog.deltadb.invoices_mor
where customer_id=102;

In [0]:
DESCRIBE HISTORY deltacatalog.deltadb.invoices_mor;

Understanding Operation Metrics :
1. numRemovedFiles: "0" 
2. numCopiedRows: "0"
3. numAddedFiles: "0"
4. numDeletedRows: "1"
5. numDeletionVectorsAdded: "1"
6. In Storage I can see two files now :
   - part-00000-439c8da6-9f56-495e-8e19-ce065170a216.c000.zstd.parquet 02/05/2026, 05:08:06 4.61 KiB
   - deletion_vector_90f76da3-0d20-46aa-ac63-28d06d9cc80f.bin          02/05/2026, 05:09:21 43 B

In [0]:
UPDATE deltacatalog.deltadb.invoices_mor
SET age=55 
where customer_id=105;

In [0]:
DESCRIBE HISTORY deltacatalog.deltadb.invoices_mor;

Understanding Operation Metrics : Update is considered as Insert and Delete
1. numRemovedFiles: "0" 
2. numCopiedRows: "0"
3. numAddedFiles: "0"
4. numDeletedRows: "1"
5. numDeletionVectorsAdded: "1"
6. numDeletionVectorsRemoved: "1"
7. numDeletionVectorsUpdated: "1"
8. In Storage I can see four files now :
   - **part-00000-439c8da6-9f56-495e-8e19-ce065170a216.c000.zstd.parquet 02/05/2026, 05:08:06 4.61 KiB**
   - deletion_vector_90f76da3-0d20-46aa-ac63-28d06d9cc80f.bin          02/05/2026, 05:09:21 43 - OBSOLETE NOW 
   - **deletion_vector_00fd5ce7-80fb-431c-9ee7-a74b18367ab2.bin          02/05/2026, 05:13:28 87** - Updated and Used
   - **part-00000-f801c37e-fcae-422b-8ef7-6f5023c11170.c000.zstd.parquet 02/05/2026, 05:13:29 2.77 KiB**

In [0]:
OPTIMIZE deltacatalog.deltadb.invoices_mor;

New File Added :
- part-00000-e60da4cc-8040-4d4a-a0ab-08635fd3651d.c000.zstd.parquet 02/05/2026, 05:19:08 4.58 KiB

In [0]:
SET spark.databricks.delta.retentionDurationCheck.enabled= false;
VACUUM deltacatalog.deltadb.invoices_mor RETAIN 0 Hours;